In [2]:
import json
from typing_extensions import TypedDict, Annotated, List
import os
from langchain_openai import ChatOpenAI
from pydantic import BaseModel
from langchain_core.callbacks import CallbackManagerForLLMRun
from typing import Optional, Any
from langchain_core.language_models.llms import LLM
import pandas as pd

data = json.load(open('data/concept_abstracts_sample.json'))

In [3]:
import json
candidate_triples = []
STEP01_OUTPUT_FILE = f'output/test/step-01.jsonl'
for line in open(STEP01_OUTPUT_FILE, 'r'):
    t = json.loads(line)
    candidate_triples.append((t['s'], t['p'], t['o']))


id_2_concept = {i: c['concept'] for i, c in
                pd.read_csv('data/refined_concepts.tsv', sep='|', header=None,
                            names=['id', 'concept'], index_col=0).iterrows()}
concept_2_id = {c: i for i, c in id_2_concept.items()}
from graphs import get_nx_graph

relation_types = json.load(open('data/relation_types.json'))
relation_2_id = {v: k for k, v in enumerate(relation_types)}

prerequisite_of_triples = []
with open('data/prerequisite-of_graph.tsv', 'r') as f:
    for line in f:
        s, p, o = line.strip().split('\t')
        prerequisite_of_triples.append((s, p, o))

graph = get_nx_graph(prerequisite_of_triples, concept_2_id, relation_2_id)

In [4]:
graph

In [5]:
from graphs import get_neighbors

print(get_neighbors(graph, 'named entity recognition', concept_2_id, id_2_concept))
print(get_neighbors(graph, 'natural language processing intro', concept_2_id, id_2_concept, mode='outgoing'))
print(get_neighbors(graph, 'named entity recognition', concept_2_id, id_2_concept, mode='ingoing'))

['linguistics basics', 'natural language processing intro']
['spelling correction', 'word sense disambiguation', 'semantic role labeling', 'chomsky hierarchy', 'named entity recognition', 'shallow parsing', 'grammar checker', 'language identification', 'information extraction', 'dialog systems', 'event detection', 'cky parsing', 'propositional logic', 'automated essay scoring', 'kernels', 'nlp for the humanities', 'semantic parsing', 'shift-reduce parsing', 'knowledge representation', 'entailment', 'machine translation', 'word embedding', 'chinese nlp', 'speech processing', 'discourse analysis', 'parsing', 'regular expressions', 'Sequence to sequence', 'sentence boundary recognition', 'document representation', 'penn treebank', 'lexicography', 'text generation', 'bio text mining', 'recommendation system', 'morphology and lexicon', 'edit distance', 'context free grammars', 'probabilistic context free grammars', 'graph-based nlp', 'sentence simplification', 'relation extraction', 'course

In [6]:
from graphs import get_2hop_neighbors

get_2hop_neighbors(graph, 'named entity recognition', concept_2_id, id_2_concept)






['spelling correction',
 'word sense disambiguation',
 'semantic role labeling',
 'chomsky hierarchy',
 'shallow parsing',
 'grammar checker',
 'language identification',
 'information extraction',
 'dialog systems',
 'event detection',
 'cky parsing',
 'propositional logic',
 'automated essay scoring',
 'kernels',
 'nlp for the humanities',
 'semantic parsing',
 'shift-reduce parsing',
 'knowledge representation',
 'entailment',
 'computational phonology',
 'chinese nlp',
 'speech processing',
 'discourse analysis',
 'prosody',
 'Sequence to sequence',
 'sentence simplification',
 'tokenization',
 'machine translation',
 'word embedding',
 'parsing',
 'regular expressions',
 'sentence boundary recognition',
 'document representation',
 'penn treebank',
 'lexicography',
 'text generation',
 'bio text mining',
 'recommendation system',
 'morphology and lexicon',
 'edit distance',
 'context free grammars',
 'probabilistic context free grammars',
 'graph-based nlp',
 'speech synthesis',
 

In [7]:
from graphs import verbalize_neighbors_triples_from_graph

print(verbalize_neighbors_triples_from_graph(graph, 'natural language processing intro', concept_2_id, id_2_concept, mode='outgoing'))
print(verbalize_neighbors_triples_from_graph(graph, 'named entity recognition', concept_2_id, id_2_concept))

(natural language processing intro,Is-a-Prerequisite-of,spelling correction)
(natural language processing intro,Is-a-Prerequisite-of,word sense disambiguation)
(natural language processing intro,Is-a-Prerequisite-of,semantic role labeling)
(natural language processing intro,Is-a-Prerequisite-of,chomsky hierarchy)
(natural language processing intro,Is-a-Prerequisite-of,named entity recognition)
(natural language processing intro,Is-a-Prerequisite-of,shallow parsing)
(natural language processing intro,Is-a-Prerequisite-of,grammar checker)
(natural language processing intro,Is-a-Prerequisite-of,language identification)
(natural language processing intro,Is-a-Prerequisite-of,information extraction)
(natural language processing intro,Is-a-Prerequisite-of,dialog systems)
(natural language processing intro,Is-a-Prerequisite-of,event detection)
(natural language processing intro,Is-a-Prerequisite-of,cky parsing)
(natural language processing intro,Is-a-Prerequisite-of,propositional logic)
(natu

In [8]:
candidate_triples

[('OCR post-correction', 'Compare', 'spelling correction'),
 ('OCR post-correction', 'Part-of', 'sequence-to-sequence model'),
 ('Neural network models', 'Compare', 'Multi-task learning'),
 ('Neural network models', 'Evaluate-for', 'Shared layers')]

In [11]:


from graphs import verbalize_neighbors_triples_from_triples

concept_name = 'OCR post-correction'
verbalize_neighbors_triples_from_triples(candidate_triples, concept_name)

('OCR post-correction', 'Compare', 'spelling correction')
('OCR post-correction', 'Part-of', 'sequence-to-sequence model')
('Neural network models', 'Compare', 'Multi-task learning')
('Neural network models', 'Evaluate-for', 'Shared layers')


'(OCR post-correction,Compare,spelling correction)\n(OCR post-correction,Part-of,sequence-to-sequence model)\n'

In [19]:
id_2_concept = {i: str(c['concept']) for i, c in
                        pd.read_csv('data/refined_concepts.tsv', sep='|', header=None,
                                    names=['id', 'concept'], index_col=0).iterrows()}
concept_2_id = {c: i for i, c in id_2_concept.items()}
with open('data/prerequisite-of_graph.tsv', 'w') as out:
    with open('data/final_new_annotation.csv', 'r') as f:
        for line in f:
            s,o,p = line.strip().split(',')
            print(s,p,o)
            if p == '1':
                print(id_2_concept[int(s)], id_2_concept[int(o)])
                out.write(f'{id_2_concept[int(s)]}\tIs-a-Prerequisite-of\t{id_2_concept[int(o)]}\n')

1 0 6
1 0 9
1 0 14
1 0 17
1 0 21
1 0 25
1 0 28
1 0 29
1 0 30
1 0 33
1 0 34
1 0 35
1 0 37
1 0 39
1 0 47
1 0 48
1 0 49
1 0 50
1 0 51
1 0 52
1 0 53
1 0 54
1 0 61
1 0 62
1 0 63
1 0 65
1 0 68
1 0 69
1 0 70
1 0 71
1 0 74
1 0 75
1 0 77
1 0 79
1 0 80
1 0 84
1 0 85
1 0 88
1 0 90
1 0 91
1 0 92
1 0 100
1 0 101
1 0 103
1 0 105
1 0 110
1 0 114
1 0 117
1 0 118
1 0 119
1 0 130
1 0 132
1 0 137
1 0 142
1 0 145
1 0 147
1 0 149
1 0 154
1 0 157
1 0 161
1 0 162
1 0 163
1 0 164
1 0 166
1 0 168
1 0 178
1 0 181
1 0 182
1 0 183
1 0 185
1 0 187
1 0 191
1 0 192
1 0 193
1 0 197
1 0 198
1 0 201
1 0 208
1 0 211
1 0 215
1 0 222
1 0 223
1 0 224
1 0 226
1 0 231
1 0 232
1 0 235
1 0 239
1 0 244
1 0 249
1 0 252
1 0 254
1 0 256
1 0 258
1 0 260
1 0 265
1 0 266
1 0 267
1 0 271
1 0 273
1 0 274
1 0 278
1 0 280
1 0 283
1 0 284
1 0 291
1 0 293
1 0 294
1 0 295
1 0 296
1 0 297
1 0 298
1 0 299
1 0 300
1 0 301
1 0 302
1 0 303
1 0 304
1 0 305
1 0 306
1 0 307
1 0 308
1 0 309
1 0 310
1 0 311
1 0 312
1 0 313
1 0 315
1 0 316
1 0 317
1 0